# LangChain Agents and Tools

### Agents
Agents in LangChain are components that utilize Language Models along with a sequence of actions to take. The agent uses a **Reasoning Edge** to determine which actions to take to get a 

### Tools
Tools are interfaces that an agent use to intereact with the world. For Example, we may use a tool to pull enterprise data and feed it into our LLM.

Agent identifies key ideas in users' question, looks at the tools at its disposal and then combined with the system prompt and the optional agent instructions, decides on which tool to use.

### ReAct Agents
**ReAct** Agents stand for Reasoning+Acting, combines reasoning and acting to solve complex language reasoning and decision-making tasks. This is achieved through text actions that the model can use to ask questions or perform tasks to gain more information and better understand a situation. 
It generally follows the sequence of Action --> Observation --> Thought --> Answer

In [1]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAI

In [2]:
load_dotenv()

True

In [3]:
model = GoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

In [4]:
from langchain.chains import LLMMathChain
from langchain.agents import Tool

In [42]:
llm_math = LLMMathChain(llm=model)
math_tool = Tool(
    name="Calculator",
    func=llm_math.run,
    description="Useful for when you need to answer questions about math. Input should be a math question.",
)

tools = [math_tool]

/Users/shailendrasharma/.local/share/virtualenvs/RAG-Hd_c4uRS/lib/python3.13/site-packages/langchain/chains/llm_math/base.py:175: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  warnings.warn(


In [9]:
print(tools[0].name,"----",tools[0].description)

Calculator ---- Useful for when you need to answer questions about math. Input should be a math question.


In [10]:
### Creating an Agent
from langchain.agents import initialize_agent, AgentType
zero_shot_agent = initialize_agent(
    agent="zero-shot-react-description",
    tools=tools,
    llm=model,
    verbose=True,
    max_iterations=10,
)


/var/folders/0q/sh8dwyl54mx5k1zrdxhrm8m80000gn/T/ipykernel_16983/1859409365.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  zero_shot_agent = initialize_agent(


In [11]:
zero_shot_agent("What is root of 25")

/var/folders/0q/sh8dwyl54mx5k1zrdxhrm8m80000gn/T/ipykernel_16983/3840991484.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  zero_shot_agent("What is root of 25")




> Entering new AgentExecutor chain...
The question asks for the square root of 25.
Action: Calculator
Action Input: sqrt(25)
Observation: Answer: 5.0
Thought:I now know the final answer
Final Answer: 5.0

> Finished chain.


{'input': 'What is root of 25', 'output': '5.0'}

In [12]:
problem = """
        You are builiding a house. There are two bedrooms of 5 meteres by 5 meters each and drawing cum open kitchen is 7 meters by 
        6 meters and balcony od 3 meteres by 2 meters.
        What is the total area of your house?
"""
zero_shot_agent(problem)



> Entering new AgentExecutor chain...
I need to calculate the area of each room and then sum them up to find the total area of the house.

Action: Calculator
Action Input: 5*5
Observation: Answer: 25
Thought:I have the area of one bedroom. Since there are two bedrooms with the same dimensions, I need to multiply this area by 2.

Action: Calculator
Action Input: 25*2
Observation: Answer: 50
Thought:I now have the total area of the two bedrooms. Now I need to calculate the area of the drawing cum open kitchen.

Action: Calculator
Action Input: 7*6
Observation: Answer: 42
Thought:I have the area of the drawing cum open kitchen. Now I need to calculate the area of the balcony.

Action: Calculator
Action Input: 3*2
Observation: Answer: 6
Thought:I have the area of the balcony. Now I need to sum up all the areas to find the total area of the house.

Action: Calculator
Action Input: 50+42+6
Observation: Answer: 98
Thought:I now know the final answer
Final Answer: 98

> Finished chain.


{'input': '\n        You are builiding a house. There are two bedrooms of 5 meteres by 5 meters each and drawing cum open kitchen is 7 meters by \n        6 meters and balcony od 3 meteres by 2 meters.\n        What is the total area of your house?\n',
 'output': '98'}

#### Using multiple tools
In the following example we will use Duck Duck Go search API as a tool combined with the Math tool.

In [15]:
from langchain.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
search_tool = Tool.from_function(
    func=search.run,
    name="Search",
    description="useful for when you need to search the interner for information"
)

llm_math_chain = LLMMathChain(llm=model, verbose=True)

math_tool = Tool.from_function(
    func=llm_math_chain.run,
    name="Calculator",
    description="Useful for when you are asked to perfrom math calculations"
)

/Users/shailendrasharma/.local/share/virtualenvs/RAG-Hd_c4uRS/lib/python3.13/site-packages/langchain/chains/llm_math/base.py:175: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  warnings.warn(


In [16]:
tools=[search_tool,math_tool]

agent = initialize_agent(
    tools,
    model,
    agent="zero-shot-react-description",
    verbose=True
)

In [18]:
agent.run("""Get Microsoft Stock Price from Internet and display in both USD and INR values""")



> Entering new AgentExecutor chain...
I need to get the Microsoft stock price in USD and then convert it to INR. I will use the search tool to find the current stock price and then the calculator to convert it to INR using the current exchange rate.
Action: Search
Action Input: "Microsoft stock price"
Observation: Get a real-time Microsoft Corporation (MSFT) stock price with an overview of financials, statistics, forecasts, charts and more. Should You Buy or Sell Microsoft Stock? Get The Latest MSFT Stock Analysis, Price Target, Dividend Info, Headlines, and Short Interest at MarketBeat. Get the latest Microsoft stock price NASDAQ: MSFT stock rating and detailed information including MSFT news, historical charts and real-time prices. Track MICROSOFT CORPORATION (MSFT) price, historical values, financial information, price forecast, and insights to empower your investing journey | MSN Money MSFT, Microsoft - Stock quote performance, technical chart analysis, SmartSelect Ratings, Group

'The Microsoft stock price is $450.18 USD, which is approximately 38364.34 INR.'

#### Creating a Custom tool

In [37]:
from langchain.tools import BaseTool
from langchain_core.output_parsers import JsonOutputParser
from langchain.tools import tool
from langchain.prompts import PromptTemplate


@tool("JSON_Response", return_direct=True)
def StructuredResponseTool(question: str):
    """
    Use this tool to send a prompt and get a JSON returned
    with three fields - Topic, Question_Details and Detailed_Response
    """
    json_prompt = PromptTemplate.from_template(
        """Return a JSON object with an `answer` key  that answers the following question: {question}. 
        The JSON object will have three fields - Topic, Question_Details and Detailed_Response"""
    )

    model = GoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.5)

    json_parser = JsonOutputParser()
    json_chain = json_prompt | model | json_parser
    x = json_chain.invoke({"question": question})
    return x


In [38]:
from langchain.agents import initialize_agent
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
tools = [StructuredResponseTool]

zero_shot_agent = initialize_agent(
    agent="zero-shot-react-description",
    tools=tools,
    llm= model,
    verbose=True,
    max_iterations=10
)

In [39]:
zero_shot_agent.run(input="Which is the top football team in the world and how?")



> Entering new AgentExecutor chain...
Determining the "top football team in the world" is subjective and depends on the criteria used. I can use the JSON_Response tool to get information about different ranking systems and their methodologies.

Action: JSON_Response
Action Input: {"Topic": "Football Team Rankings", "Question_Details": "Explain the different methods used to rank football teams globally, including FIFA rankings, Elo ratings, and other notable ranking systems. Discuss the strengths and weaknesses of each method and how they determine the top team.", "Detailed_Response": "Provide a detailed explanation of various football team ranking systems and their methodologies."}
Observation: {'Topic': 'Football Team Rankings', 'Question_Details': 'Explain the different methods used to rank football teams globally, including FIFA rankings, Elo ratings, and other notable ranking systems. Discuss the strengths and weaknesses of each method and how they determine the top team.', 'Deta

{'Topic': 'Football Team Rankings',
 'Question_Details': 'Explain the different methods used to rank football teams globally, including FIFA rankings, Elo ratings, and other notable ranking systems. Discuss the strengths and weaknesses of each method and how they determine the top team.',
 'Detailed_Response': {'answer': 'Several methods are used to rank football teams globally, each with its own methodology, strengths, and weaknesses. Here\'s an overview of some prominent systems:\n\n**1. FIFA Rankings:**\n\n*   **Methodology:** The FIFA rankings use a points-based system that considers match results over the past four years. More recent matches, more important matches (e.g., World Cup matches vs. friendlies), and matches against higher-ranked opponents are weighted more heavily. The calculation involves adding and subtracting points based on the outcome of a match. The formula considers the relative strength of the two teams, the importance of the match, and the expected outcome.\n\n

#### Conversational ReAct Agents
This agent is designed for use in conversational settings. It utilizes the ReAct framework to determine which tool to use and utilizes memory to remember previous conversation interactions. 

In [40]:
from langchain.agents import create_react_agent, load_tools
from langchain.memory import ConversationBufferMemory

In [47]:
tools = [math_tool]

conversational_agent = initialize_agent(
    agent="conversational-react-description",
    tools=tools,
    llm=model,
    verbose=True,
    max_iterations=10,
    memory=ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
)

/var/folders/0q/sh8dwyl54mx5k1zrdxhrm8m80000gn/T/ipykernel_16983/3853287791.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(


In [48]:
output1=conversational_agent.run("Add 7 to 9 and tell me the result")
output2=conversational_agent.run("Add 5 to the previous result")



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 7+9
Observation: Answer: 16
Thought:Do I need to use a tool? No
AI: The result of adding 7 to 9 is 16.
```

> Finished chain.


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 16+5
Observation: Answer: 21
Thought:Do I need to use a tool? No
AI: The result of adding 5 to the previous result (16) is 21.

> Finished chain.


#### ReAct DocStore
This agent utilizes the reAct framework to communicate with a docstore. It requires the availability of a Search tool and a Lookup tool with identical names. The Search tool is used search for a document, while the Lookup tool looks up a term within the most recently found document.

A `docstore` in Langchain is essentially a storage system for document. It provides a way for langchain agemts to access and interact with information.

In [52]:
from langchain.agents import initialize_agent, Tool
from langchain import Wikipedia
from langchain.agents.react.base import DocstoreExplorer

In [54]:
docstore = DocstoreExplorer(Wikipedia())
tools = [
    Tool(
        name="Search", func=docstore.search, description="Useful for when you need to search wikipedia"
        ),
    Tool(
        name="Lookup",
        func=docstore.lookup,
        description="Useful for when you need to lookup a specific term in wikipedia"
    )
]

llm = GoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
docstore_agent = initialize_agent(
    agent="react-docstore",
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=10,
    handle_parse_errors=True
)

/var/folders/0q/sh8dwyl54mx5k1zrdxhrm8m80000gn/T/ipykernel_16983/2356968821.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  docstore = DocstoreExplorer(Wikipedia())


In [55]:
docstore_agent("Tell me a few things about Maharana Pratap")



> Entering new AgentExecutor chain...
Thought: I need to search Maharana Pratap and summarize a few things about him.
Action: Search[Maharana Pratap]
Observation: Could not find [Maharana Pratap]. Similar: ['Maharana Pratap', 'Pratap Jayanti', 'Bharat Ka Veer Putra – Maharana Pratap', 'Maharana Pratap Airport', 'Maharana Pratap Sports College', 'Battle of Haldighati', 'Maharana Pratap Sagar', 'Udaipur', 'Maharana Pratap Inter State Bus Terminus', 'Vishal Jethwa']
Thought:I will search for "Maharana Pratap" again.
Action: Search["Maharana Pratap"]
Observation: Pratap Singh I (9 May 1540 – 19 January 1597), popularly known as Maharana Pratap (IPA: [məɦaːˈɾaːɳaː pɾəˈtaːp] ), was king of the Kingdom of Mewar, in north-western India in the present-day state of Rajasthan, from 1572 until his death in 1597. He is notable for leading the Rajput resistance against the expansionist policy of the Mughal Emperor Akbar including the battle of Haldighati and the battle of Dewair.
Thought:Maharana 

{'input': 'Tell me a few things about Maharana Pratap',
 'output': 'Maharana Pratap was the king of Mewar from 1572 to 1597. He is known for leading the Rajput resistance against Mughal Emperor Akbar. He led the battle of Haldighati and the battle of Dewair.'}

#### Self ask with Search (Self Ask Agents)
The **Self-Ask** approach allows for an LLM to give answers to a question it was no explicitly trained on. The model might not have the direct answer to a question, but answers to sub-questions will exist in the LLM dataset. It achieves this by obtaining Intermediate answers.



In [56]:
search= DuckDuckGoSearchRun()
tools = [
    Tool(
        func=search.run,
        name="Intermediate Answer",
        description="Useful for when you need to search internet for answers"
    )
]
agent = initialize_agent(
    agent="self-ask-with-search",
    tools=tools,
    llm=llm,
    verbose=True,
    handle_parsing_errors=True,
)


In [57]:
print(agent.invoke("Question: Who was the president of USA when first moon landing happened? "))



> Entering new AgentExecutor chain...
Yes.
Follow up: When did the first moon landing happen?
Intermediate answer: Watch highlights from the Apollo 11 mission including the launch on July 16, 1969, the landing of the lunar module, Neil Armstrong's first steps on the Moon, splashdown, and more. Watch on NASA+ 20 July 1969—Astronaut Edwin E. Aldrin Jr., lunar module pilot, walks on the surface of the moon near the leg of the Lunar Module (LM) "Eagle ... Witness Neil Armstrong take humankind's first steps on the lunar surface during NASA's Apollo 11 mission Perhaps the most famous of all space films, these clips document the arrival of the first human beings on the Moon during the afternoon of July 20, 1969. They comprise footage of the landing of the Apollo 11 spacecraft, taken with a 16-mm camera mounted in Edwin Aldrin's window, and Neil ... Soon after that the first Moon landing and the first landing on any extraterrestrial body was performed by Luna 2, [1] which intentionally impac